# Phase 5: Definitive method sweep — ranked by competition RMSE on TVT

After falsifying naive GR-window matching across 773 wells, we grounded in what
the leaderboard actually does (public scores ~9.3; a Grandmaster XGB starter ~15;
top public kernels are **DWT/wavelet-based**) and in the geosteering physics
(TVT = log projected along the dipping surface, with stretch/squeeze; an *anchor*
point where correction is zero). This notebook tests every viable method family
as a comparable arm and **ranks them purely by RMSE on TVT** — the competition
metric — under the honest 73% forward-tail CV.

### Why a sweep (and why these arms)
Local single-well tests are inconclusive (our one dev well is flat, where the
floor is near-optimal). The real contest is on **high-movement wells**, which
only exist at scale. So this runs across all wells and reports RMSE overall and
by movement bucket. Arms, by hypothesis:

- **Floor** — hold last TVT (the bar; ~13 on this data).
- **Regression** — XGB/LightGBM/HGB on *relative* features (matches the ~15 starter).
- **Raw-GR matching** — the falsified baseline, kept to confirm it stays bad.
- **Smoothed / multi-scale matching** — match the *coarse* GR trend (we measured
  broad-trend corr ≈ +0.77 vs ≈ 0 for raw windows). The principled core of DWT.
- **Wavelet (DWT) matching** — denoise GR to coarse scales, then match.
- **Dynamic warping (banded)** — stretch/squeeze alignment per the geosteering physics.
- **Hybrids** — matcher as a feature feeding a residual model over the floor.

> **Honesty note.** As built, on the single local well *nothing* beats the floor
> (that well is flat). The point of this notebook is to find what beats the floor
> **on the high-movement bucket** when run on your full data. That bucket is the
> experiment that decides the approach.

## Setup

In [ ]:
import warnings

warnings.filterwarnings("ignore")
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.ndimage import uniform_filter1d

sys.path.insert(0, str(Path("../src").resolve()))
from rogii_wellbore import clean  # noqa: E402

# optional libs (arms self-skip if missing)
try:
    import pywt

    HAVE_PYWT = True
except Exception:
    HAVE_PYWT = False
try:
    from dtaidistance import dtw

    HAVE_DTW = True
except Exception:
    HAVE_DTW = False
try:
    import lightgbm as lgb

    HAVE_LGB = True
except Exception:
    HAVE_LGB = False
try:
    import xgboost as xgb

    HAVE_XGB = True
except Exception:
    HAVE_XGB = False
from sklearn.ensemble import HistGradientBoostingRegressor

cfg = clean.load_config("../data/interim/clean_config.json")
CLEAN_DIR = Path("../data/interim/clean")
REAL_EVAL_FRAC = 0.73
print(f"pywt={HAVE_PYWT} dtw={HAVE_DTW} lgb={HAVE_LGB} xgb={HAVE_XGB}")

In [ ]:
def tail_mask(n, frac):
    k = round(n * frac)
    m = np.zeros(n, bool)
    if k:
        m[n - k :] = True
    return m


def rmse(a, b):
    return float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2)))


def load_pair(wid):
    hz = (
        pd.read_csv(CLEAN_DIR / "train" / f"{wid}__horizontal_well.csv", dtype={"well_id": str})
        .sort_values("MD")
        .reset_index(drop=True)
    )
    tw = pd.read_csv(
        CLEAN_DIR / "train" / f"{wid}__typewell.csv", dtype={"well_id": str}
    ).reset_index(drop=True)
    return hz, tw


WELLS = sorted(p.name.split("__")[0] for p in (CLEAN_DIR / "train").glob("*__horizontal_well.csv"))
print(f"{len(WELLS)} train wells")

## The arms

Each arm is a function `(hz, tw, kn, ev) -> pred_tvt` over the eval rows. They all
share the same masking and scoring so RMSE-on-TVT is directly comparable.

In [ ]:
def _prep(hz, tw, frac):
    tvt = hz["TVT"].values.astype(float)
    Z = hz["Z"].values.astype(float)
    MD = hz["MD"].values.astype(float)
    GRz = hz["GR_z"].values.astype(float)
    m = tail_mask(len(hz), frac)
    kn = np.where(~m)[0]
    ev = np.where(m)[0]
    twt = tw["TVT"].values.astype(float)
    twg = tw["GR_z"].values.astype(float)
    return tvt, Z, MD, GRz, kn, ev, twt, twg


# ---- match core (shared by raw/smoothed/wavelet) ----
def _match_on_signal(lat_sig_by_tvt, fine, tw_sig, twt, prior, ev_len, search_ft, win_ft):
    grid = np.arange(-win_ft, win_ft + 1e-9, 0.5)
    out = np.empty(ev_len)
    cand_mask = (twt >= prior - search_ft) & (twt <= prior + search_ft)
    cand = twt[cand_mask]
    lp = np.interp(prior + grid, fine, lat_sig_by_tvt)
    if np.std(lp) < 1e-9 or len(cand) == 0:
        out[:] = prior
        return out
    best_c, best_t = -2.0, prior
    for c0 in cand:
        seg = np.interp(c0 + grid, twt, tw_sig)
        if np.std(seg) < 1e-9:
            continue
        cc = np.corrcoef(lp, seg)[0, 1]
        if cc > best_c:
            best_c, best_t = cc, c0
    # NOTE: this returns ONE refined center; per-row variation handled by callers
    out[:] = best_t
    return out


def _lateral_signal_by_tvt(tvt, GRz, kn, transform):
    order = np.argsort(tvt)
    lt, lg = tvt[order], GRz[order]
    g = np.isfinite(lg)
    lt, lg = lt[g], lg[g]
    if len(lt) < 5:
        return None, None
    fine = np.arange(lt.min(), lt.max(), 0.5)
    sig = np.interp(fine, lt, lg)
    return transform(sig), fine


# ---------- ARMS ----------
def arm_floor(hz, tw, frac):
    tvt, _Z, _MD, _GRz, kn, ev, _twt, _twg = _prep(hz, tw, frac)
    return np.full(len(ev), tvt[kn[-1]]), tvt[ev]


def arm_match_raw(hz, tw, frac, search_ft=40, win_ft=12):
    tvt, Z, MD, GRz, kn, ev, twt, twg = _prep(hz, tw, frac)
    prior = tvt[kn[-1]]
    lat, fine = _lateral_signal_by_tvt(tvt, GRz, kn, lambda s: s)
    if lat is None:
        return np.full(len(ev), prior), tvt[ev]
    p = _match_on_signal(lat, fine, twg, twt, prior, len(ev), search_ft, win_ft)
    return p, tvt[ev]


def arm_match_smooth(hz, tw, frac, smooth_ft=10, search_ft=40, win_ft=20):
    tvt, Z, MD, GRz, kn, ev, twt, twg = _prep(hz, tw, frac)
    prior = tvt[kn[-1]]
    w = max(1, int(smooth_ft / 0.5))
    tw_sm = uniform_filter1d(twg, w)
    lat, fine = _lateral_signal_by_tvt(tvt, GRz, kn, lambda s: uniform_filter1d(s, w))
    if lat is None:
        return np.full(len(ev), prior), tvt[ev]
    p = _match_on_signal(lat, fine, tw_sm, twt, prior, len(ev), search_ft, win_ft)
    return p, tvt[ev]


def arm_match_wavelet(hz, tw, frac, search_ft=40, win_ft=20, wavelet="db4"):
    if not HAVE_PYWT:
        return None, None
    tvt, Z, MD, GRz, kn, ev, twt, twg = _prep(hz, tw, frac)
    prior = tvt[kn[-1]]

    def denoise(sig):
        sig = np.nan_to_num(
            sig, nan=np.nanmedian(sig[np.isfinite(sig)]) if np.isfinite(sig).any() else 0.0
        )
        lvl = min(3, pywt.dwt_max_level(len(sig), wavelet))
        if lvl < 1:
            return sig
        c = pywt.wavedec(sig, wavelet, level=lvl)
        for i in range(len(c) - 1, max(0, len(c) - 2), -1):
            c[i] = np.zeros_like(c[i])
        return pywt.waverec(c, wavelet)[: len(sig)]

    tw_dn = denoise(twg)
    lat, fine = _lateral_signal_by_tvt(tvt, GRz, kn, denoise)
    if lat is None:
        return np.full(len(ev), prior), tvt[ev]
    p = _match_on_signal(lat, fine, tw_dn, twt, prior, len(ev), search_ft, win_ft)
    return p, tvt[ev]


def arm_dtw_band(hz, tw, frac, band_ft=40):
    if not HAVE_DTW:
        return None, None
    tvt, _Z, _MD, GRz, kn, ev, twt, twg = _prep(hz, tw, frac)
    prior = tvt[kn[-1]]
    band = np.where(np.abs(twt - prior) <= band_ft)[0]
    if len(band) < 10:
        return np.full(len(ev), prior), tvt[ev]
    lat_ev = np.nan_to_num(GRz[ev], nan=np.nanmedian(GRz[kn]))
    from collections import defaultdict

    path = dtw.warping_path(lat_ev, twg[band])
    mp = defaultdict(list)
    for li, ti in path:
        mp[li].append(twt[band][ti])
    p = np.array([np.mean(mp[k]) if k in mp else prior for k in range(len(ev))])
    return p, tvt[ev]


def _rel_feats(hz, idx, kn):
    Z = hz["Z"].values.astype(float)
    MD = hz["MD"].values.astype(float)
    GRz = hz["GR_z"].values.astype(float)
    dZ = np.gradient(Z)
    dMD = np.gradient(MD)
    incl = dZ / np.where(dMD == 0, 1, dMD)
    dGR = np.gradient(np.nan_to_num(GRz))
    grw = uniform_filter1d(np.nan_to_num(GRz), 10)
    return np.c_[np.nan_to_num(GRz[idx]), incl[idx], dGR[idx], grw[idx], Z[idx] - Z[kn[-1]]]


def arm_regression(hz, tw, frac, kind="hgb"):
    tvt, Z, MD, GRz, kn, ev, twt, twg = _prep(hz, tw, frac)
    Xtr, ytr = _rel_feats(hz, kn, kn), tvt[kn]
    if kind == "hgb":
        mdl = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.05)
    elif kind == "lgb" and HAVE_LGB:
        mdl = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, num_leaves=63, verbose=-1)
    elif kind == "xgb" and HAVE_XGB:
        mdl = xgb.XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6, verbosity=0)
    else:
        return None, None
    mdl.fit(Xtr, ytr)
    return mdl.predict(_rel_feats(hz, ev, kn)), tvt[ev]


def arm_hybrid_match_resid(hz, tw, frac, smooth_ft=10):
    # Matcher output as a feature; HGB predicts TVT from [match, floor, rel-feats].
    # Lets the model learn when to trust the match vs the floor.
    tvt, Z, MD, GRz, kn, ev, twt, twg = _prep(hz, tw, frac)
    prior_all = tvt[kn[-1]]
    w = max(1, int(smooth_ft / 0.5))
    tw_sm = uniform_filter1d(twg, w)

    # match feature per row: nearest smoothed-GR TVT within +-40 of a causal prior
    def match_feat(idx, prior_val):
        grz_sm = uniform_filter1d(np.nan_to_num(GRz, nan=np.nanmedian(GRz[kn])), w)
        out = []
        for i in idx:
            cand = np.where(np.abs(twt - prior_val) <= 40)[0]
            if len(cand) == 0:
                out.append(prior_val)
                continue
            out.append(twt[cand[np.argmin(np.abs(tw_sm[cand] - grz_sm[i]))]])
        return np.array(out)

    # train with causal prior = previous TVT (known zone); eval prior = last known
    mf_tr = match_feat(kn[1:], tvt[kn[0]])  # crude causal prior
    Xtr = np.c_[_rel_feats(hz, kn[1:], kn), mf_tr]
    ytr = tvt[kn[1:]]
    mdl = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.05).fit(Xtr, ytr)
    mf_ev = match_feat(ev, prior_all)
    Xev = np.c_[_rel_feats(hz, ev, kn), mf_ev]
    return mdl.predict(Xev), tvt[ev]

## Run the sweep across all wells

In [ ]:
ARMS = {
    "floor": lambda hz, tw: arm_floor(hz, tw, REAL_EVAL_FRAC),
    "reg_hgb": lambda hz, tw: arm_regression(hz, tw, REAL_EVAL_FRAC, "hgb"),
    "reg_lgb": lambda hz, tw: arm_regression(hz, tw, REAL_EVAL_FRAC, "lgb"),
    "reg_xgb": lambda hz, tw: arm_regression(hz, tw, REAL_EVAL_FRAC, "xgb"),
    "match_raw": lambda hz, tw: arm_match_raw(hz, tw, REAL_EVAL_FRAC),
    "match_smooth10": lambda hz, tw: arm_match_smooth(hz, tw, REAL_EVAL_FRAC, 10),
    "match_smooth30": lambda hz, tw: arm_match_smooth(hz, tw, REAL_EVAL_FRAC, 30),
    "match_wavelet": lambda hz, tw: arm_match_wavelet(hz, tw, REAL_EVAL_FRAC),
    "dtw_band": lambda hz, tw: arm_dtw_band(hz, tw, REAL_EVAL_FRAC),
    "hybrid_match": lambda hz, tw: arm_hybrid_match_resid(hz, tw, REAL_EVAL_FRAC),
}


def run_sweep(wells, n=None):
    wells = wells[:n] if n else wells
    rows = []
    for w_i, wid in enumerate(wells):
        try:
            hz, tw = load_pair(wid)
        except Exception:
            continue
        if "TVT" not in hz or hz["TVT"].isna().all():
            continue
        nrow = len(hz)
        if round(nrow * REAL_EVAL_FRAC) < 20 or nrow - round(nrow * REAL_EVAL_FRAC) < 20:
            continue
        tvt = hz["TVT"].values.astype(float)
        m = tail_mask(nrow, REAL_EVAL_FRAC)
        span = float(tvt[m].max() - tvt[m].min())
        rec = {"well": wid, "eval_span": span}
        for name, fn in ARMS.items():
            try:
                p, t = fn(hz, tw)
                rec[name] = rmse(p, t) if p is not None else np.nan
            except Exception:
                rec[name] = np.nan
        rows.append(rec)
        if (w_i + 1) % 50 == 0:
            print(f"  ...{w_i+1} wells")
    return pd.DataFrame(rows)


# START SMALL to confirm timing, then set n=None for the full run.
sweep = run_sweep(WELLS)
print(f"swept {len(sweep)} wells")

## Ranking — RMSE on TVT (the competition metric)

In [ ]:
arm_cols = [c for c in sweep.columns if c not in ("well", "eval_span")]

overall = sweep[arm_cols].mean().sort_values()
print("=== OVERALL mean RMSE on TVT (lower = better) ===")
floor_v = overall.get("floor", np.nan)
for k, v in overall.items():
    tag = "" if k == "floor" else (" <<< beats floor" if v < floor_v else "")
    print(f"  {k:16s} {v:8.3f}{tag}")
print(f"\n(public LB leaders ~9.3; XGB starter ~15; floor here = {floor_v:.2f})")

In [ ]:
# By movement bucket — the decisive view. The HIGH bucket is where the real
# contest lives (floor is weak there); whatever wins HIGH is the approach.
print("=== mean RMSE by eval-span bucket ===")
for lo, hi, lab in [
    (0, 5, "FLAT <5ft"),
    (5, 15, "MED 5-15"),
    (15, 40, "HIGH 15-40"),
    (40, 1e9, "XHIGH >40"),
]:
    sub = sweep[(sweep.eval_span >= lo) & (sweep.eval_span < hi)]
    if not len(sub):
        continue
    mt = sub[arm_cols].mean().sort_values()
    fv = mt.get("floor", np.nan)
    print(f"\n  [{lab}] n={len(sub)}  floor={fv:.2f}")
    for k, v in mt.head(5).items():
        tag = "" if k == "floor" else (" <<<" if v < fv else "")
        print(f"     {k:16s} {v:7.3f}{tag}")

In [ ]:
# Win-rate: per well, which arm has the lowest RMSE? (robustness beyond the mean)
valid = sweep[arm_cols].dropna(axis=1, how="all")
winner = valid.idxmin(axis=1)
print("=== per-well winner counts ===")
print(winner.value_counts())
print(f"\nfloor win-rate: {(winner == 'floor').mean():.0%}")
print("(if a matching/hybrid arm wins often on HIGH-span wells, that is the path)")

## Blow-up diagnostics — why matching fails on some wells

The sweep shows matching wins ~70% of wells by win-rate but loses on mean (it
blows up on the rest). These cells compute, per well, predictors that are
measurable WITHOUT truth, then correlate them with whether matching actually
beat the floor. The predictor that best separates wins from blow-ups is the
**gate** a confidence-hybrid should use (match when safe, fall back to floor when not).

Predictors:
- `known_corr` — corr(lateral GR, typewell GR at the same TVT) in the KNOWN zone.
  Low ⇒ the typewell is a poor reference for this well ⇒ matching will fail.
- `tw_band_std` — typewell GR variability in the search band. Low ⇒ featureless
  reference ⇒ search wanders.
- `match_disp` / `match_conf` — how far the best match lands from the prior, and
  its correlation. Large jump + low corr ⇒ untrustworthy.

In [ ]:
def well_diagnostics(hz, tw, frac=REAL_EVAL_FRAC, smooth_ft=20, search_ft=40, win_ft=20):
    """Per-well failure predictors (no truth) + truth-based outcome."""
    tvt = hz["TVT"].values.astype(float)
    GRz = hz["GR_z"].values.astype(float)
    Z = hz["Z"].values.astype(float)  # noqa: F841
    MD = hz["MD"].values.astype(float)  # noqa: F841
    m = tail_mask(len(hz), frac)
    kn = np.where(~m)[0]
    ev = np.where(m)[0]
    twt = tw["TVT"].values.astype(float)
    twg = tw["GR_z"].values.astype(float)
    prior = tvt[kn[-1]]
    tw_at_known = np.interp(tvt[kn], twt, twg)
    gk = GRz[kn]
    good = np.isfinite(gk)
    known_corr = np.corrcoef(gk[good], tw_at_known[good])[0, 1] if good.sum() > 5 else np.nan
    band = np.where(np.abs(twt - prior) <= search_ft)[0]
    tw_band_std = float(twg[band].std()) if len(band) else 0.0
    w = max(1, int(smooth_ft / 0.5))
    tw_sm = uniform_filter1d(twg, w)
    order = np.argsort(tvt)
    lt, lg = tvt[order], GRz[order]
    g = np.isfinite(lg)
    lt, lg = lt[g], lg[g]
    if len(lt) > 5:
        fine = np.arange(lt.min(), lt.max(), 0.5)
        lg_sm = uniform_filter1d(np.interp(fine, lt, lg), w)
        grid = np.arange(-win_ft, win_ft + 1e-9, 0.5)
        lp = np.interp(prior + grid, fine, lg_sm)
        cand = twt[(twt >= prior - search_ft) & (twt <= prior + search_ft)]
        bc, bt = -2.0, prior
        for c0 in cand:
            seg = np.interp(c0 + grid, twt, tw_sm)
            if np.std(seg) < 1e-9:
                continue
            cc = np.corrcoef(lp, seg)[0, 1]
            if cc > bc:
                bc, bt = cc, c0
        match_disp, match_conf, match_pred = bt - prior, bc, bt
    else:
        match_disp, match_conf, match_pred = 0.0, -2.0, prior
    floor_rmse = rmse(prior, tvt[ev])
    match_rmse = rmse(match_pred, tvt[ev])
    return dict(
        known_corr=known_corr,
        tw_band_std=tw_band_std,
        match_disp=match_disp,
        abs_match_disp=abs(match_disp),
        match_conf=match_conf,
        eval_span=float(tvt[ev].max() - tvt[ev].min()),
        floor_rmse=floor_rmse,
        match_rmse=match_rmse,
        match_beats_floor=int(match_rmse < floor_rmse),
    )


diag_rows = []
for wid in WELLS:  # full set; matches the sweep
    try:
        hz, tw = load_pair(wid)
    except Exception:
        continue
    if "TVT" not in hz or hz["TVT"].isna().all():
        continue
    nrow = len(hz)
    if round(nrow * REAL_EVAL_FRAC) < 20 or nrow - round(nrow * REAL_EVAL_FRAC) < 20:
        continue
    try:
        d = well_diagnostics(hz, tw)
        d["well"] = wid
        diag_rows.append(d)
    except Exception:
        continue
diagdf = pd.DataFrame(diag_rows)
print(f"diagnostics on {len(diagdf)} wells")
print(
    diagdf[["known_corr", "tw_band_std", "abs_match_disp", "match_conf", "match_beats_floor"]]
    .describe()
    .round(3)
)

In [ ]:
# Correlate each no-truth predictor with whether matching beat the floor.
preds = ["known_corr", "tw_band_std", "abs_match_disp", "match_conf", "eval_span"]
print("=== predictor vs match-success (point-biserial corr) ===")
for p in preds:
    sub = diagdf[[p, "match_beats_floor"]].dropna()
    if len(sub) > 5:
        c = np.corrcoef(sub[p], sub["match_beats_floor"])[0, 1]
        print(f"  {p:16s} corr_with_win = {c:+.3f}")

print("\n=== predictor means: wins vs blow-ups ===")
print(diagdf.groupby("match_beats_floor")[preds].mean().round(3))

# Simulate a gated hybrid: use match where predictor passes a threshold, else floor.
print("\n=== gated hybrid simulation (match if known_corr>thr, else floor) ===")
for thr in [0.3, 0.5, 0.7]:
    use = diagdf["known_corr"] > thr
    blended = np.where(use, diagdf["match_rmse"], diagdf["floor_rmse"])
    print(
        f"  known_corr>{thr}: mean RMSE {np.mean(blended):.2f} "
        f"(floor={diagdf['floor_rmse'].mean():.2f}, pure-match={diagdf['match_rmse'].mean():.2f}, "
        f"use {use.mean():.0%} of wells)"
    )

In [ ]:
# ORACLE SELECTOR: if we could pick the best arm per well, what would we score?
# This is the ceiling for the per-well-selection architecture the public
# "Forced Selector" kernel (7.631) appears to use.
oracle_sel = sweep[arm_cols].min(axis=1)
print(f"floor mean:            {sweep['floor'].mean():.2f}")
print(f"ORACLE selector mean:  {oracle_sel.mean():.2f}")
print("\nby bucket:")
for lo, hi, lab in [(0, 5, "FLAT"), (5, 15, "MED"), (15, 40, "HIGH"), (40, 1e9, "XHIGH")]:
    sub = sweep[(sweep.eval_span >= lo) & (sweep.eval_span < hi)]
    if len(sub):
        print(
            f"  [{lab:5s}] n={len(sub):3d}  floor={sub['floor'].mean():6.2f}  "
            f"oracle_sel={sub[arm_cols].min(axis=1).mean():6.2f}"
        )
# How often is each arm the per-well pick? (already have winner counts, but
# also show the GAP: how much does the winner beat the floor by, per well?)
gap = sweep["floor"] - oracle_sel
print(
    f"\nper-well improvement over floor: mean {gap.mean():.2f} ft, "
    f"median {gap.median():.2f}, p90 {gap.quantile(0.9):.2f}"
)

## How to read this, and what to do next

**Ranking is by RMSE on TVT — the competition metric — nothing else.**

- **OVERALL table**: blunt average. The floor will be hard to beat overall because
  most wells are easy; don't over-index on this.
- **By-bucket table**: the real signal. On **HIGH / XHIGH** spans the floor is weak
  (~14+). Whatever arm beats the floor *there* is the mechanism to pursue.
- **Win-rate**: robustness — an arm that wins on many wells (not just mean) is
  safer to build on.

**Decision rule:** pick the arm (or hybrid) that most consistently beats the floor
on HIGH-span wells, then (a) tune its config (smooth width, search width, wavelet
level), and (b) wrap it in the `hybrid_match` style so a learned model blends it
with the floor per row — predicting TVT directly, scored on this same harness.

Run the full set first: change `run_sweep(WELLS, n=40)` to `run_sweep(WELLS)`.
Expect the matching arms to take the most time (the per-row search); if too slow,
lower the number of `cand` by tightening `search_ft`, or subsample eval rows for
the sweep and re-score the winner densely.

Whatever wins here is what notebook 6 builds into a full submission pipeline,
scored against the public-LB reference points (~9.3 leaders, ~15 starter).